In [ ]:
# BOOKCORPUS — BOOK BOUNDARY DETECTION + CLEAN EXTRACTION

from datasets import load_dataset
import re
import os
import zipfile
import hashlib
import random
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token



SEED          = 42
MAX_BOOKS     = 10000
MIN_SENTENCES = 1500
END_ZONE      = 100             # only check soft signals in last N sentences of a book
OUTPUT_DIR    = "books_output"
ZIP_PATH      = "books_output.zip"
TARGET_TOKENS = 300_000_000
current_tokens = 0

quote_skipped = 0
repetition_skipped = 0
noise_skipped = 0



random.seed(SEED)


# STEP 1: LOAD DATASET
print("Loading dataset...")
dataset = load_dataset("rojagtap/bookcorpus", split="train",streaming=True)


# STEP 2: BOUNDARY PATTERNS
hard_boundary_patterns = [
    re.compile(r"isbn[\s:]+[\d\-]+",              re.IGNORECASE),
    re.compile(r"copyright\s+\d{4}",              re.IGNORECASE),
    re.compile(r"smashwords edition",              re.IGNORECASE),
    re.compile(r"published by .+ on smashwords",   re.IGNORECASE),
    re.compile(r"all rights reserved",             re.IGNORECASE),
    re.compile(r"this ebook is licensed",          re.IGNORECASE),
    re.compile(r"this ebook may not be resold",    re.IGNORECASE),
    re.compile(r"this ebook may not be re-sold",   re.IGNORECASE),
    re.compile(r"licen[sc]ed for your personal",   re.IGNORECASE),
    re.compile(r"printed in the u\.s\.a",          re.IGNORECASE),
    re.compile(r"table of contents",               re.IGNORECASE),
    re.compile(r"rave reviews for",                re.IGNORECASE),
    re.compile(r"this is a sampler collection",    re.IGNORECASE),
]

soft_boundary_patterns = [
    re.compile(r"follow me on",                    re.IGNORECASE),
    re.compile(r"join my email list",              re.IGNORECASE),
    re.compile(r"click here to sign up",           re.IGNORECASE),
    re.compile(r"about the author",                re.IGNORECASE),
    re.compile(r"acknowledgements",                re.IGNORECASE),
    re.compile(r"acknowledgments",                 re.IGNORECASE),
    re.compile(r"back to table of contents",       re.IGNORECASE),
    re.compile(r"go back to table of contents",    re.IGNORECASE),
    re.compile(r"find me at",                      re.IGNORECASE),
    re.compile(r"you can contact",                 re.IGNORECASE),
    re.compile(r"contact me via",                  re.IGNORECASE),
    re.compile(r"please consider writing a review",re.IGNORECASE),
    re.compile(r"your reviews and referrals",      re.IGNORECASE),
    re.compile(r"available for purchase on",       re.IGNORECASE),
    re.compile(r"find these books on",             re.IGNORECASE),
    re.compile(r"goodreads facebook blog twitter", re.IGNORECASE),
]

email_pattern  = re.compile(
    r"[\w.\-]{1,50}@[\w.\-]{1,50}\.(com|net|org|edu|gmail|hotmail|yahoo)",
    re.IGNORECASE
)
url_pattern    = re.compile(r"https?://\S{1,200}|www\.\S{1,200}", re.IGNORECASE)
mailto_pattern = re.compile(r"mailto:", re.IGNORECASE)


def is_boundary(text, current_book_len):
    if not text or len(text.strip()) < 3:
        return False, None
    for pattern in hard_boundary_patterns:
        if pattern.search(text):
            return True, pattern.pattern
    if current_book_len <= END_ZONE:
        for pattern in soft_boundary_patterns:
            if pattern.search(text):
                return True, pattern.pattern
        if email_pattern.search(text):
            return True, "email"
        if url_pattern.search(text):
            return True, "url"
        if mailto_pattern.search(text):
            return True, "mailto"
    return False, None

def count_tokens(book):
    total = 0
    for sentence in book:
        total += len(tokenizer.encode(sentence))
    return total

def is_noisy_book(book_sentences):
    text = " ".join(book_sentences)
    if not text:
        return True

    total = len(text)

    weird = sum(1 for c in text if not (c.isalnum() or c in " .,!?'-\""))
    weird_ratio = weird / total

    punct = sum(1 for c in text if c in ".,!?")
    punct_ratio = punct / total

    # Too many weird symbols
    if weird_ratio > 0.05: # more than 5% non-standard chars → probably garbage
        return True

    # Too little punctuation (bad structure)
    if punct_ratio < 0.003: # less than 0.3% punctuation → probably not prose
        return True

    return False



# STEP 3: PURE SAMPLER DETECTION

# Only reject the entire book if it has NO real story content —
# i.e. it is entirely a sampler/preview collection with nothing
# but chapter samples stitched together by promo text.
# If it has story content, the inline scrub handles stray lines.

pure_sampler_patterns = [
    re.compile(r"sampler collection",                           re.IGNORECASE),
    re.compile(r"this is a sampler",                            re.IGNORECASE),
    re.compile(r"samplings? (are )?brought to you",             re.IGNORECASE),
    re.compile(r"samples? (are )?from the following",           re.IGNORECASE),
    re.compile(r"preview collection",                           re.IGNORECASE),
    re.compile(r"first \d+ chapters? of several",               re.IGNORECASE),
    re.compile(r"collection of (excerpts|samples|previews)",    re.IGNORECASE),
]

# These are sampler transition lines that appear between sample
# chapters — their presence means the book is a multi-book sampler
sampler_transition_patterns = [
    re.compile(r"you have finished (the )?(first |last )?\d+ chapters?", re.IGNORECASE),
    re.compile(r"you have finished (reading |the )?sampling",   re.IGNORECASE),
    re.compile(r"please continue (reading |the )?sampling",     re.IGNORECASE),
    re.compile(r"please continue reading samples?",             re.IGNORECASE),
]

def is_pure_sampler(book_sentences):
    """
    Only returns True if the book is definitively a multi-book sampler
    with NO real story content of its own. Checks for:
    1. Explicit sampler declaration in first 50 lines, OR
    2. Multiple sampler transition markers (meaning multiple sample
       chapters from different books are stitched together).
    A single stray line is NOT enough to reject the whole book.
    """
    # Check first 50 lines for explicit sampler declarations
    for sent in book_sentences[:50]:
        for pattern in pure_sampler_patterns:
            if pattern.search(sent):
                return True

    # Count sampler transition markers across the whole book
    # More than 1 transition = definitely a multi-book sampler
    transition_count = 0
    for sent in book_sentences:
        for pattern in sampler_transition_patterns:
            if pattern.search(sent):
                transition_count += 1
                break
        if transition_count > 1:
            return True

    return False

def normalize_quotes_and_spacing(text):
    import re

    # Fix contractions: i 'm → i'm
    text = re.sub(r"\b(\w)\s+'\s*(\w)", r"\1'\2", text)

    # Fix n't: ca n't → can't
    text = re.sub(r"\b(\w+)\s+n['’]\s*t\b", r"\1n't", text)

    # Replace fancy quotes
    text = text.replace("``", '"').replace("''", '"')

    # Remove excessive quotes
    text = re.sub(r"[\"`]{3,}", '"', text)
    text = re.sub(r"[']{3,}", "'", text)

    # Fix spacing around punctuation
    text = re.sub(r"\s+([.,!?])", r"\1", text)
    text = re.sub(r"([.,!?])([^\s])", r"\1 \2", text)

    # Remove repeated punctuation
    text = re.sub(r"[.,!?]{2,}", ".", text)

    return text.strip()
    



# STEP 4: SENTENCE-LEVEL METADATA FILTERS
# PHILOSOPHY: Only remove sentences that are UNAMBIGUOUSLY non-story.
# We do NOT remove sentences just because they contain app names,
# website names, or words that could appear in real prose.
# We only remove lines where the ENTIRE sentence is clearly metadata.

# These are safe to remove anywhere — they cannot be story content
unambiguous_non_story = [
    # Hard legal boilerplate — entire sentence is a legal notice
    re.compile(r"^all rights reserved[\.\s]*$",                 re.IGNORECASE),
    re.compile(r"no part of this (book|publication|work) may",  re.IGNORECASE),
    re.compile(r"no portion of this (book|publication|work) may",re.IGNORECASE),
    re.compile(r"reproduction is not permitted",                 re.IGNORECASE),
    re.compile(r"this is a work of fiction[\.\s]*$",            re.IGNORECASE),
    re.compile(r"licen[sc]ed for your personal enjoyment",       re.IGNORECASE),
    re.compile(r"this ebook (is licensed|may not be)",          re.IGNORECASE),
    re.compile(r"thank you for respecting the hard work",        re.IGNORECASE),

    # Sampler transition lines — unambiguously non-story
    re.compile(r"you have finished (the )?(first |last )?\d+ chapters?", re.IGNORECASE),
    re.compile(r"you have finished (reading |the )?sampling",   re.IGNORECASE),
    re.compile(r"please continue (reading |the )?sampling",     re.IGNORECASE),
    re.compile(r"please continue reading samples?",             re.IGNORECASE),
    re.compile(r"for continued reading.{0,40}(purchase|buy|visit)", re.IGNORECASE),
    re.compile(r"this book is available in paperback",          re.IGNORECASE),

    # Rave review block markers
    re.compile(r"^rave reviews for\b",                          re.IGNORECASE),
    re.compile(r"the kindle book review",                       re.IGNORECASE),

    # Author sign-off lines
    re.compile(r"^dear reader ,",                               re.IGNORECASE),
    re.compile(r"your (humble )?(servant|author) ,",            re.IGNORECASE),
    re.compile(r"enjoy this humble offering",                   re.IGNORECASE),

    # TOC navigation links
    re.compile(r"^(go )?back to table of contents[\.\s]*$",     re.IGNORECASE),
    re.compile(r"^link to table of contents[\.\s]*$",           re.IGNORECASE),

    # Explicit purchase/promo lines (entire sentence is promo)
    re.compile(r"^if you.?d like to (purchase|find|buy).{0,60}(smashwords|amazon|author)", re.IGNORECASE),
    re.compile(r"^please (go to|visit) (smashwords|amazon|the author)", re.IGNORECASE),
    re.compile(r"^(click|tap) here to (purchase|buy|download|sign up)", re.IGNORECASE),
    re.compile(r"you have (now )?reached the end of",           re.IGNORECASE),
]

stray_punct_pattern = re.compile(r"^[\W_]+$")

def is_unambiguous_non_story(text):
    """
    Returns True ONLY for sentences that are unambiguously non-story.
    Conservative — does NOT remove sentences just because they contain
    website names, app names, or words that could appear in real prose.
    """
    stripped = text.strip()
    if not stripped or stray_punct_pattern.match(stripped):
        return True
    # Must match a URL or email that occupies most of the sentence
    if url_pattern.search(text) and len(text.strip()) < 120:
        # Only remove if the sentence is mostly a URL (not story prose
        # that happens to contain a URL the character is reading)
        words = text.split()
        url_words = sum(1 for w in words if url_pattern.search(w))
        if url_words / max(len(words), 1) > 0.4:
            return True
    for pattern in unambiguous_non_story:
        if pattern.search(text):
            return True
    return False

def has_too_many_quotes(book_sentences, max_ratio=0.10):
    text = " ".join(book_sentences)

    quote_count = sum(text.count(q) for q in ['"', "'", '`'])
    total = len(text)

    if total == 0:
        return True

    weird_patterns = len(re.findall(r"``|''", text))

    ratio = (quote_count + 3 * weird_patterns) / total

    return ratio > max_ratio

def is_quote_heavy(book_sentences, max_ratio=0.40):
    total = len(book_sentences)
    if total == 0:
        return True

    quote_sentences = sum(
        1 for s in book_sentences if any(q in s for q in ['"', "'", '`'])
    )

    return (quote_sentences / total) > max_ratio   

def has_repetition(book_sentences, max_repeat_ratio=0.25):
    from collections import Counter

    words = " ".join(book_sentences).split()
    if not words:
        return True

    counts = Counter(words)
    most_common = counts.most_common(1)[0][1]

    return (most_common / len(words)) > max_repeat_ratio
    

# STEP 5: METADATA STRIPPING (3-pass)

# Header signals — used only for the HEAD scan (first 200 lines)
# These are safe to use as head-skip signals because we're only
# scanning the pre-story block, not the whole book body
header_signals = [
    "ebook may not", "all rights reserved", "no part of this",
    "no portion of this", "smashwords", "copyright",
    "isbn", "this book is licensed", "may not be re-sold",
    "may not be resold", "names , characters , places",
    "any resemblance", "dedicated to",
    "in memory of", "in loving memory", "table of contents",
    "thank you for respecting", "reproduction is not permitted",
    "no part of this publication", "rights reserved",
    "printed in the u.s.a", "first published",
    "rave reviews", "a note from the author",
    "note from the author", "for continued reading",
    "sampler collection", "prelude",
    "link to table of contents",
]

# End-matter sentinels — used only for the TAIL scan (last 30%)
# These are safe because we only look at the tail zone, not the
# whole book body, so false positives in the story are avoided
end_matter_sentinels = [
    re.compile(r"^about the author",                            re.IGNORECASE),
    re.compile(r"authors? (history|bio|biography|profile)",     re.IGNORECASE),
    re.compile(r"^acknowledgements?\s*$",                       re.IGNORECASE),
    re.compile(r"^acknowledgments?\s*$",                        re.IGNORECASE),
    re.compile(r"^author'?s? note\s*$",                         re.IGNORECASE),
    re.compile(r"^a note from the author",                      re.IGNORECASE),
    re.compile(r"^rave reviews",                                re.IGNORECASE),
    re.compile(r"^other (books|titles|works) by\b",             re.IGNORECASE),
    re.compile(r"^also by\b",                                   re.IGNORECASE),
    re.compile(r"^books by\b",                                  re.IGNORECASE),
    re.compile(r"sneak peek",                                   re.IGNORECASE),
    re.compile(r"^follow me on\b",                              re.IGNORECASE),
    re.compile(r"join my (mailing|email) list",                 re.IGNORECASE),
    re.compile(r"please consider writing a review",             re.IGNORECASE),
    re.compile(r"^dear reader ,",                               re.IGNORECASE),
    re.compile(r"^link to table of contents",                   re.IGNORECASE),
    re.compile(r"thank you for (reading|downloading|purchasing) (my|this|the)", re.IGNORECASE),
    re.compile(r"^if you enjoyed (this|the) (book|story|novel|series)", re.IGNORECASE),
    re.compile(r"^(please )?check out (my|our) other books",   re.IGNORECASE),
    re.compile(r"you have (now )?reached the end",              re.IGNORECASE),
    re.compile(r"for continued reading",                        re.IGNORECASE),
    re.compile(r"non.fiction novels? include",                  re.IGNORECASE),
    re.compile(r"(he|she) is the (acclaimed|bestselling|award.winning|celebrated) author", re.IGNORECASE),
    re.compile(r"lives (full.time|part.time) (on|in|at)",      re.IGNORECASE),
]

def is_end_matter(text):
    for pattern in end_matter_sentinels:
        if pattern.search(text):
            return True
    return False


story_start_signals = [
    re.compile(r"^(chapter|prologue|part)\s+(one|two|three|\d+)", re.IGNORECASE),
    re.compile(r"^(it was|he was|she was|they were|i was|the \w+)", re.IGNORECASE),
    re.compile(r"^(``|'').+said",                                re.IGNORECASE),
]

def find_story_start(book_sentences, max_scan=200):
    """
    Find where the story actually begins by scanning the first max_scan
    lines. Advances past header-signal lines. Returns when it hits a
    clear story-start marker after at least 5 lines of metadata.
    """
    best = 0
    for i, sent in enumerate(book_sentences[:max_scan]):
        lower = sent.lower()
        if any(sig in lower for sig in header_signals):
            best = i + 1
        for pat in story_start_signals:
            if pat.match(sent.strip()) and i > 5:
                return i
    return best


def strip_book_metadata(book_sentences):
    """
    3-pass cleaning — conservative philosophy:

    Pass 1 — HEAD: Skip extended pre-story blocks (legal, TOC, reviews,
             author notes) by finding where the actual story starts.
             Only affects the first 200 lines.

    Pass 2 — TAIL: Cut from the first end-matter sentinel found in the
             last 30% of the book (author bio, promo, series listing).
             Only affects the tail zone — never touches the story body.

    Pass 3 — INLINE SCRUB: Remove only UNAMBIGUOUSLY non-story sentences
             (legal boilerplate, sampler transitions, explicit promo lines).
             Does NOT remove sentences just because they contain app/website
             names — only removes sentences where the entire sentence is
             clearly metadata, not story content.
    """
    if not book_sentences:
        return book_sentences

    # Pass 1 — head
    story_start = find_story_start(book_sentences, max_scan=200)
    book_sentences = book_sentences[story_start:]

    # Pass 2 — tail
    tail_window = max(0, len(book_sentences) - min(400, len(book_sentences) * 30 // 100))
    cut_at = len(book_sentences)
    for j in range(tail_window, len(book_sentences)):
        if is_end_matter(book_sentences[j]):
            cut_at = j
            break
    book_sentences = book_sentences[:cut_at]

    # Pass 3 — conservative inline scrub
    cleaned = []
    for s in book_sentences:
        if not is_unambiguous_non_story(s):
            s = normalize_quotes_and_spacing(s)  
            cleaned.append(s)

    return cleaned


# STEP 6: DEDUPLICATION

def get_book_fingerprint(book_sentences):
    fingerprint = " ".join(book_sentences[:5]).lower().strip()
    fingerprint = re.sub(r'\s+', ' ', fingerprint)
    return hashlib.md5(fingerprint.encode()).hexdigest()


# STEP 7: SCAN DATASET FOR BOOK BOUNDARIES

print(f"\nScanning for first {MAX_BOOKS} valid books...")
print("=" * 60)

books                 = []
current_book          = []
seen_hashes           = set()
fragments_skipped     = 0
duplicates_skipped    = 0
too_short_after_clean = 0
samplers_skipped      = 0
random_skipped = 0

for i, example in enumerate(dataset):
    text = example["text"]
    hit_boundary, hit = is_boundary(text, len(current_book))

    if hit_boundary:
        if len(current_book) >= MIN_SENTENCES:
        
            # Reject pure samplers
            if is_pure_sampler(current_book):
                samplers_skipped += 1
                current_book = []
                continue
        
            clean_book = strip_book_metadata(current_book)

            
            if has_too_many_quotes(clean_book) and is_quote_heavy(clean_book):
                print(f"\n❌ Skipped (quote-heavy) ({len(clean_book)} sentences)")
                quote_skipped += 1
                current_book = []
                continue
            
            if has_repetition(clean_book):
                print(f"\n❌ Skipped (repetition) ({len(clean_book)} sentences)")
                repetition_skipped += 1
                current_book = []
                continue
            
            if is_noisy_book(clean_book):
                print(f"\n❌ Skipped (noisy) ({len(clean_book)} sentences)")
                noise_skipped += 1
                current_book = []
                continue
            # Random skip (diversity)
            if random.random() < 0.20:   # (recommended value)
                print(f"\n🎲 Skipped (random diversity) ({len(clean_book)} sentences)")
                random_skipped+=1
                current_book = []
                continue
                
            if len(clean_book) >= MIN_SENTENCES:
                fp = get_book_fingerprint(clean_book)
        
                if fp not in seen_hashes:
        
                    tokens = count_tokens(clean_book)
        
                    # Token budget check
                    if current_tokens + tokens > TARGET_TOKENS:
                        print("\n✅ Reached ~400M tokens. Stopping.")
                        break
        
                    seen_hashes.add(fp)
                    books.append(clean_book)
                    current_tokens += tokens
                    
                    PREVIEW_LINES = 5  # how many lines to show at start/end
                    
                    print(f"\n📚 Book {len(books)} detected!")
                    print(f"   Raw sentences   : {len(current_book):,}")
                    print(f"   Clean sentences : {len(clean_book):,}")
                    print(f"   Tokens          : {tokens:,}")

                    if len(books) >= MAX_BOOKS:
                        print(f"\n✅ Found {MAX_BOOKS} books — stopping scan.")
                        break
                else:
                    duplicates_skipped += 1
                    print(f"\n🔁 Duplicate skipped ({len(clean_book):,} sentences)")
            else:
                too_short_after_clean += 1
                print(f"\n⚠️  Too short after cleaning "
                      f"({len(clean_book)} sentences) — skipped")
        else:
            if len(current_book) > 10:
                fragments_skipped += 1
                print(f"\n⚠️  Fragment skipped ({len(current_book)} sentences)")

        current_book = []
    else:
        current_book.append(text)
        


if len(current_book) >= MIN_SENTENCES:

    if not is_pure_sampler(current_book):

        clean_book = strip_book_metadata(current_book)

        if not is_noisy_book(clean_book):

            if has_too_many_quotes(clean_book) and is_quote_heavy(clean_book):
                print(f"\n❌ Skipped (quote-heavy FINAL) ({len(clean_book)} sentences)")
                quote_skipped += 1
            
            elif has_repetition(clean_book):
                print(f"\n❌ Skipped (repetition FINAL) ({len(clean_book)} sentences)")
                repetition_skipped += 1
            
            elif is_noisy_book(clean_book):
                print(f"\n❌ Skipped (noisy FINAL) ({len(clean_book)} sentences)")
                noise_skipped += 1
            
            else:
                if len(clean_book) >= MIN_SENTENCES:
                    fp = get_book_fingerprint(clean_book)
            
                    if fp not in seen_hashes:
                        tokens = count_tokens(clean_book)
            
                        if current_tokens + tokens <= TARGET_TOKENS:
                            seen_hashes.add(fp)
                            books.append(clean_book)
                            current_tokens += tokens
            
                            print(f"\n📚 Final book added")
                            print(f"   Tokens so far: {current_tokens/1e6:.1f}M")


COMBINED_FILE = "all_books_300M.txt"

print(f"\nSaving all books into one file: {COMBINED_FILE}")

with open(COMBINED_FILE, "w", encoding="utf-8") as f:

    for idx, book in enumerate(books):

        # # 🔹 Book header (added to check the dataset but commented so that it wont affect training)
        # f.write(f"\n{'#'*40}\n")
        # f.write(f"BOOK {idx+1}\n")
        # f.write(f"{'#'*40}\n\n")

        # 🔹 Book content
        for sentence in book:
            f.write(sentence.strip() + "\n")

        # 🔹 Extra spacing between books
        f.write("\n\n")

print(f"✅ Saved {len(books)} books into {COMBINED_FILE}")

print("\n📊 FILTER STATS")
print(f"Quote-heavy skipped: {quote_skipped}")
print(f"Repetition skipped : {repetition_skipped}")
print(f"Noisy skipped      : {noise_skipped}")
print(f"Random skipped      : {random_skipped}")